# Problem Sheet 3 — Yang Cheng, yacheng@mpia.de

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py
from scipy.stats import binned_statistic_dd
import illustris_python as il

## 1. Evolution of density contrast on different spatial scales

### 1.1

I use TNG100-3-Dark. First I pick the snapshots closest to z = 11, 5, 2, 0.

In [11]:
basePath = '/home/tnguser/sims.TNG/TNG100-3-Dark/output'

redshifts = np.array([il.groupcat.loadHeader(basePath, snap)['Redshift'] for snap in range(100)])
snaps = [int(np.argmin(np.abs(redshifts - z))) for z in [11, 5, 2, 0]]

for snap in snaps:
    print(f'snap {snap}: z = {redshifts[snap]:.2f}')

#the DM particle mass is only in the snapshot header (MassTable), same as in sheet 1
with h5py.File(basePath + '/snapdir_%03d/snap_%03d.%s.hdf5' % (99, 99, 0), 'r') as f:
    header = dict(f['Header'].attrs)
h = header['HubbleParam']
dm_mass = header['MassTable'][1] * 1e10 / h        # Msun per particle
print(f'Particle mass: {dm_mass:.3e} Msun')

snap 3: z = 10.98
snap 17: z = 5.00
snap 33: z = 2.00
snap 99: z = 0.00
Particle mass: 5.668e+08 Msun


In [ ]:
#I manually set the hist bins to 256, for both the 2d and 3d grid
nbins = 256
rho_crit0 = 2.775e11 * h**2                        # Msun/Mpc^3


mean_rho_2d = []
mean_rho_3d = []
for snap in snaps:
    header = il.groupcat.loadHeader(basePath, snap)
    a = header['Time']
    z = header['Redshift']
    box = header['BoxSize'] * a / h / 1000         # Mpc

    Coordinates = il.snapshot.loadSubset(basePath, snap, partType=1, fields=['Coordinates']) * a / h / 1000

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    pixel_area = (box / nbins)**2
    weights = np.full(len(Coordinates), dm_mass / pixel_area)
    Sigma, xedges, yedges, image = axes[0].hist2d(x=Coordinates[:, 0], y=Coordinates[:, 1], weights=weights,
        bins=nbins, range=[[0, box], [0, box]], norm=mpl.colors.LogNorm(), cmap='magma')
    axes[0].set_title(f'z = {z:.1f}: column density')
    fig.colorbar(image, ax=axes[0], label=r'$\Sigma$ [$M_\odot$/Mpc$^2$]', location='bottom')

    counts = binned_statistic_dd(Coordinates, np.arange(len(Coordinates)), bins=nbins,
        range=[[0, box]] * 3, statistic='count')[0]
    rho = counts * dm_mass / (box / nbins)**3
    rho_avg = len(Coordinates) * dm_mass / box**3

    image = axes[1].imshow(np.ma.masked_less_equal(rho[:, :, nbins // 2].T, 0), origin='lower',
        extent=[0, box, 0, box], norm=mpl.colors.LogNorm(), cmap='magma')
    axes[1].set_title(f'z = {z:.1f}: 3D density, slice at z = {box / 2:.1f} Mpc')
    fig.colorbar(image, ax=axes[1], label=r'$\rho$ [$M_\odot$/Mpc$^3$]', location='bottom')

    for ax in axes:
        ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]')
        ax.set_aspect('equal')
    plt.show()

    print(f'z = {z:.2f}, box = {box:.2f} Mpc, cell = {box / nbins * 1000:.0f} kpc')
    print(f'  Sigma: {Sigma[Sigma > 0].min():.2e} to {Sigma.max():.2e} Msun/Mpc^2, median {np.median(Sigma):.2e}')
    print(f'  rho:   {rho[rho > 0].min():.2e} to {rho.max():.2e} Msun/Mpc^3')
    print(f'  mean rho = {rho_avg:.3e} Msun/Mpc^3, Omega_m rho_crit0 (1+z)^3 = {header["Omega0"] * rho_crit0 * (1 + z)**3:.3e}')

### 1.1 discussion

At high z the maps are almost uniform, and going to z=0 the matter collapses into filaments and haloes with big voids in between, so the range of densities gets much wider. The mean density of the box is just total mass over physical volume, so it goes like $(1+z)^3$, and it agrees with $\Omega_m\rho_{crit,0}(1+z)^3$ (the total mass is conserved, only the physical volume changes).

### 1.2

$\delta = (\rho - \bar\rho)/\bar\rho$, measured on 3D cubes of 10 Mpc, 1 Mpc and 100 kpc at z=0, using `binned_statistic_dd` as in the input notebook. For 100 kpc the full box would need $1107^3$ cells, which doesn't fit in memory, so there I only take a 20 Mpc sub-cube with all the particles.

In [ ]:
snap = 99
header = il.groupcat.loadHeader(basePath, snap)
a = header['Time']
box = header['BoxSize'] * a / h / 1000             # Mpc

Coordinates = il.snapshot.loadSubset(basePath, snap, partType=1, fields=['Coordinates']) * a / h / 1000
rho_avg = len(Coordinates) * dm_mass / box**3

def density_contrast(Coordinates, box, nbins, mass, rho_avg):
    counts = binned_statistic_dd(Coordinates, np.arange(len(Coordinates)), bins=nbins,
        range=[[0, box]] * 3, statistic='count')[0].flatten()
    rho = counts * mass / (box / nbins)**3
    return rho / rho_avg - 1

deltas = {}
for cell, n_downsample in [(10, 100), (1, 5)]:
    deltas[cell] = density_contrast(Coordinates[rng.random(len(Coordinates)) < 1 / n_downsample], box, int(box / cell),
                                    dm_mass * n_downsample, rho_avg)

sub = 20                                           # Mpc
inside = np.all(Coordinates < sub, axis=1)
deltas[0.1] = density_contrast(Coordinates[inside], sub, int(sub / 0.1), dm_mass, rho_avg)

bins = np.linspace(-2, 6, 81)
fig, ax = plt.subplots(figsize=(8, 6))
for cell, delta in deltas.items():
    ax.hist(np.log10(1 + delta[delta > -1]), bins=bins, density=True, histtype='step', label=f'{cell} Mpc')
    print(f'{cell} Mpc: max delta = {delta.max():.1f}, empty cells = {np.mean(delta == -1):.1%}')
ax.set(xlabel=r'$\log_{10}(1+\delta)$', ylabel='PDF', yscale='log', title='z = 0')
ax.legend()
plt.show()

### 1.2 discussion

The PDF is very asymmetric: $\delta$ can't go below $-1$ (empty cells) but has a long tail to high values, so it looks roughly log-normal rather than Gaussian. The smaller the smoothing scale, the wider the PDF and the higher the maximum $\delta$, since small cells can sit on a single halo while 10 Mpc cells average over haloes and voids. The 100 kpc case is below the mean interparticle spacing of TNG100-3-Dark ($110.7/455 \approx 0.24$ Mpc), so most cells are empty and the PDF there is mostly particle shot noise.

### 1.3

Same function at z ≈ 11, 10, 5, 2, 0. I keep the same comoving cell size (65 cells per side, i.e. 7 lattice cells or 1.7 cMpc) at all redshifts, so the same material is compared.

In [ ]:
nbins = 65
n_downsample = 5
snaps_13 = [int(np.argmin(np.abs(redshifts - z))) for z in [11, 10, 5, 2, 0]]

fig, ax = plt.subplots(figsize=(8, 6))
for snap in snaps_13:
    header = il.groupcat.loadHeader(basePath, snap)
    a = header['Time']
    z = header['Redshift']
    box = header['BoxSize'] * a / h / 1000         # Mpc

    Coordinates = il.snapshot.loadSubset(basePath, snap, partType=1, fields=['Coordinates'])
    Coordinates = Coordinates[rng.random(len(Coordinates)) < 1 / n_downsample] * a / h / 1000
    mass = dm_mass * n_downsample
    rho_avg = len(Coordinates) * mass / box**3

    delta = density_contrast(Coordinates, box, nbins, mass, rho_avg)
    ax.hist(np.log10(1 + delta[delta > -1]), bins=np.linspace(-2, 3, 61), density=True,
            histtype='step', label=f'z = {z:.1f}')
    print(f'z = {z:.1f}: max delta = {delta.max():.1f}')
ax.set(xlabel=r'$\log_{10}(1+\delta)$', ylabel='PDF', yscale='log', title='1.7 cMpc cells')
ax.legend()
plt.show()

### 1.3 discussion

At z > 10 the PDF is a narrow peak around $\delta = 0$, the matter is still almost uniform as in the initial conditions. With time gravity makes overdense regions denser and underdense ones emptier, so the PDF gets wider on both sides and the high-density tail grows a lot, while the peak moves slightly below $\delta = 0$ because most of the volume ends up in voids.

### 1.4

TNG100-3-Dark vs TNG300-3-Dark at z=0, both with 3 Mpc cells. TNG300 has a lower resolution, so I need bigger cells to have enough particles per cell.

In [ ]:
cell = 3                                           # Mpc
sims = [('TNG100-3-Dark', basePath, 10),
        ('TNG300-3-Dark', '/home/tnguser/sims.TNG/TNG300-3-Dark/output', 4)]

fig, ax = plt.subplots(figsize=(8, 6))
for name, path, n_downsample in sims:
    with h5py.File(path + '/snapdir_%03d/snap_%03d.%s.hdf5' % (99, 99, 0), 'r') as f:
        header = dict(f['Header'].attrs)
    box = header['BoxSize'] / h / 1000             # Mpc, a = 1
    mass = header['MassTable'][1] * 1e10 / h * n_downsample

    #float32 because TNG300 has 625^3 particles
    Coordinates = il.snapshot.loadSubset(path, 99, partType=1, fields=['Coordinates'], float32=True)
    Coordinates = Coordinates[rng.random(len(Coordinates)) < 1 / n_downsample] / h / 1000
    rho_avg = len(Coordinates) * mass / box**3

    delta = density_contrast(Coordinates, box, int(box / cell), mass, rho_avg)
    ax.hist(np.log10(1 + delta[delta > -1]), bins=np.linspace(-2, 4, 61), density=True,
            histtype='step', label=name)
    print(f'{name}: box = {box:.1f} Mpc, max delta = {delta.max():.1f}')
ax.set(xlabel=r'$\log_{10}(1+\delta)$', ylabel='PDF', yscale='log', title=f'z = 0, {cell} Mpc cells')
ax.legend()
plt.show()

### 1.4 discussion

The two PDFs agree in the bulk, since the smoothing scale is the same, but TNG300 extends further in the high-density tail and has a higher maximum $\delta$. The most massive clusters are very rare, and the 27 times larger volume of TNG300 simply contains more (and more extreme) of them, so the small box can't sample the tail of the distribution.

## 2. Gravitational collapse and the formation of haloes

### 2.1

Cluster with $M = 10^{15}\,M_\odot$ inside $R = 2$ Mpc, compared to the mean matter density today $\bar\rho = \Omega_m\rho_{crit,0}$, with $\rho_{crit,0} = 2.775\times10^{11}h^2\,M_\odot/\mathrm{Mpc}^3$ and the TNG cosmology.

In [ ]:
M_cluster = 1e15                                   # Msun
R_cluster = 2                                      # Mpc
rho_cluster = M_cluster / (4 / 3 * np.pi * R_cluster**3)

Om0 = 0.3089
rho_mean = Om0 * 2.775e11 * h**2                   # Msun/Mpc^3

delta_cluster = rho_cluster / rho_mean - 1
print(f'rho_cluster = {rho_cluster:.3e} Msun/Mpc^3')
print(f'rho_mean    = {rho_mean:.3e} Msun/Mpc^3')
print(f'delta = {delta_cluster:.0f}, log10(1+delta) = {np.log10(1 + delta_cluster):.2f}')

### 2.1 discussion

$\delta \approx 760$, i.e. $\log_{10}(1+\delta) \approx 2.9$. On similar scales (1–3 Mpc cells) this is at the very end of the tails of the PDFs in 1.2 and 1.4, only a tiny fraction of cells reach it, so such clusters are very rare objects, and a big box like TNG300 is needed to find more than a handful of them.

## 3. The distribution of material within dark-matter haloes

### 3.1

For the halo profiles I use TNG50-4-Dark, as suggested in the input notebook, since it has a better mass resolution than TNG100-3-Dark. Radial bins are log-spaced from $0.03R_{200c}$ to $R_{200c}$.

In [ ]:
basePath_50 = '/home/tnguser/sims.TNG/TNG50-4-Dark/output'
snap = 99

with h5py.File(basePath_50 + '/snapdir_%03d/snap_%03d.%s.hdf5' % (snap, snap, 0), 'r') as f:
    header = dict(f['Header'].attrs)
a = header['Time']
box = header['BoxSize']                            # ckpc/h, for the periodic wrap
pmass = header['MassTable'][1] * 1e10 / h          # Msun per particle

halos = il.groupcat.loadHalos(basePath_50, snap, fields=['GroupPos', 'Group_M_Crit200', 'Group_R_Crit200'])
M200c = halos['Group_M_Crit200'] * 1e10 / h        # Msun
R200c = halos['Group_R_Crit200'] * a / h           # kpc

def distances(Coordinates, GroupPos, BoxSize):
    d = Coordinates - GroupPos
    d = (d + BoxSize / 2) % BoxSize - BoxSize / 2  # periodic box
    return np.linalg.norm(d, axis=1) * a / h       # kpc

def density_profile(haloID, n_bins=20):
    Coordinates = il.snapshot.loadHalo(basePath_50, snap, haloID, partType=1, fields=['Coordinates'])
    r = distances(Coordinates, halos['GroupPos'][haloID], box)
    bins = np.logspace(np.log10(0.03 * R200c[haloID]), np.log10(R200c[haloID]), n_bins + 1)
    mass = np.histogram(r, bins=bins)[0] * pmass
    volume = 4 / 3 * np.pi * (bins[1:]**3 - bins[:-1]**3)
    return np.sqrt(bins[1:] * bins[:-1]), mass / volume

haloID = int(np.argmin(np.abs(M200c - 1e12)))
r, rho = density_profile(haloID)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(r, rho, marker='o')
ax.set(xscale='log', yscale='log', xlabel='r [kpc]', ylabel=r'$\rho$ [$M_\odot$/kpc$^3$]',
       title=f'Halo {haloID}: M200c = {M200c[haloID]:.2e} Msun, R200c = {R200c[haloID]:.0f} kpc')
plt.show()

### 3.1 discussion

The profile falls by several orders of magnitude from the centre to $R_{200c}$. In log-log it is not a single power law: it is shallower in the centre (about $r^{-1}$) and steepens towards $r^{-3}$ in the outskirts, which is the NFW shape.

### 3.2

Mass bins of 0.2 dex around $10^{11}$, $10^{12}$ and $10^{13}\,M_\odot$ (at most 50 haloes per bin), mean profile per bin, vs $r$ and vs $r/R_{200c}$. Since the bins go from $0.03R_{200c}$ to $R_{200c}$ for every halo, the $r/R_{200c}$ grid is the same for all of them.

In [ ]:
mass_bins = [1e11, 1e12, 1e13]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
for M in mass_bins:
    select = np.where((M200c > M * 10**-0.1) & (M200c < M * 10**0.1))[0][:50]
    profiles = [density_profile(i) for i in select]
    r_mean = np.mean([p[0] for p in profiles], axis=0)
    rho_mean = np.mean([p[1] for p in profiles], axis=0)
    x = profiles[0][0] / R200c[select[0]]

    axes[0].plot(r_mean, rho_mean, marker='o', label=f'M200c ~ {M:.0e} Msun (N = {len(select)})')
    axes[1].plot(x, rho_mean, marker='o', label=f'M200c ~ {M:.0e} Msun')
    print(f'M200c ~ {M:.0e}: N = {len(select)}, rho at {r_mean[0]:.1f} kpc = {rho_mean[0]:.2e} Msun/kpc^3')

axes[0].set(xscale='log', yscale='log', xlabel='r [kpc]', ylabel=r'$\rho$ [$M_\odot$/kpc$^3$]')
axes[1].set(xscale='log', xlabel=r'$r/R_{200c}$')
for ax in axes:
    ax.legend()
plt.show()

### 3.2 discussion

Vs physical radius, more massive haloes are bigger and reach higher densities at the same $r$, and their innermost bin sits at larger $r$. Vs $r/R_{200c}$ the three profiles almost fall on top of each other: all haloes have the same mean density inside $R_{200c}$ ($200\rho_{crit}$ by definition), so the profiles are nearly self-similar. The small differences left are in the shape, lower-mass haloes being a bit more concentrated.

### 3.3

NFW fit, $\rho(r) = \rho_0 / [(r/r_s)(1+r/r_s)^2]$, for 300 random haloes with $M_{200c} > 10^{11}\,M_\odot$. I fit $\log_{10}\rho$, otherwise the inner bins dominate the fit.

In [ ]:
from scipy.optimize import curve_fit

def log_NFW(r, rs, log_rho0):
    x = r / rs
    return log_rho0 - np.log10(x * (1 + x)**2)

rng = np.random.default_rng(0)
sample = np.where(M200c > 1e11)[0]
sample = rng.choice(sample, size=min(300, len(sample)), replace=False)

rs_fit = []
for i in sample:
    r, rho = density_profile(i)
    good = rho > 0
    popt = curve_fit(log_NFW, r[good], np.log10(rho[good]), p0=[R200c[i] / 5, 7],
                     bounds=([0.01 * R200c[i], 0], [R200c[i], 12]))[0]
    rs_fit.append(popt[0])
rs_fit = np.array(rs_fit)

slope, offset = np.polyfit(np.log10(M200c[sample]), np.log10(rs_fit), 1)
print(f'r_s ~ M200c^{slope:.2f}')

M_plot = np.logspace(11, np.log10(M200c[sample].max()), 50)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(M200c[sample], rs_fit, s=8)
ax.plot(M_plot, 10**offset * M_plot**slope, color='black', label=f'slope = {slope:.2f}')
ax.set(xscale='log', yscale='log', xlabel=r'$M_{200c}$ [$M_\odot$]', ylabel=r'$r_s$ [kpc]')
ax.legend()
plt.show()

### 3.3 discussion

$r_s$ grows with halo mass roughly as a power law. $R_{200c} \propto M_{200c}^{1/3}$, so a slope a bit steeper than 1/3 means the concentration $c = R_{200c}/r_s$ decreases slowly with mass: massive haloes formed later, when the universe was less dense, and are less concentrated.

## 4. Build your own Python Cosmological Calculator (Part II)

Growth factor with the Carroll+1992 fitting formula:

$g(\Omega_m,\Omega_\Lambda) = \dfrac{5}{2}\Omega_m \Big/ \left[\Omega_m^{4/7} - \Omega_\Lambda + \left(1+\dfrac{\Omega_m}{2}\right)\left(1+\dfrac{\Omega_\Lambda}{70}\right)\right]$

$D_+(z) = \dfrac{1}{1+z}\,\dfrac{g(\Omega_m(z),\Omega_\Lambda(z))}{g(\Omega_{m,0},\Omega_{\Lambda,0})}$

with $\Omega_m(z)$, $\Omega_\Lambda(z)$ from $E(z)$ as in Problem Sheet 2, so that $D_+(0) = 1$.

In [ ]:
def E(z, Om, Or, OL):
    Ok = 1 - Om - Or - OL
    return np.sqrt(Or*(1+z)**4 + Om*(1+z)**3 + Ok*(1+z)**2 + OL)

def g_carroll(Om, OL):
    return 2.5*Om / (Om**(4/7) - OL + (1+Om/2)*(1+OL/70))

def D_plus(z, Om, Or, OL):
    Omz = Om*(1+z)**3 / E(z, Om, Or, OL)**2
    OLz = OL / E(z, Om, Or, OL)**2
    return g_carroll(Omz, OLz) / g_carroll(Om, OL) / (1+z)

print('EdS D+(z=1) =', D_plus(1, 1, 0, 0))

In [ ]:
z = np.linspace(0, 10, 200)
models = {
    'EdS':                          (1.0, 0, 0),
    'Low density (Om=0.3, OL=0)':   (0.3, 0, 0),
    'Standard (Om=0.3, OL=0.7)':    (0.3, 0, 0.7),
}

fig, ax = plt.subplots(figsize=(8, 6))
for name, (Om, Or, OL) in models.items():
    ax.plot(z, D_plus(z, Om, Or, OL), label=name)
ax.set(xlabel='z', ylabel=r'$D_+(z)$', yscale='log', title='Linear growth factor')
ax.legend()
plt.show()

### 4 discussion

For EdS $D_+ = 1/(1+z)$ exactly. The other two models also start at $D_+(0) = 1$ but decrease more slowly towards high z: with less matter (or with $\Lambda$) the growth of structure is suppressed at late times, so to reach the same amplitude today the fluctuations had to be larger in the past. The low-density model is the most extreme, LCDM is in between. At high z $\Omega_m(z) \to 1$ in all models, so they all go like $(1+z)^{-1}$ with constant offsets relative to EdS ($\approx 2.2$ for low density and $\approx 1.29$ for LCDM).